# Phase 1: Data Understanding and Exploratory Analysis

This notebook profiles the UCI Online Retail transaction dataset, documents data quality issues, and saves the required EDA figures. All paths are relative to the project root and all results are reproducible with seed 42.

In [1]:
from pathlib import Path
import hashlib
import json
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style='whitegrid', palette='deep')

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'Online Retail.xlsx'
FIGURES_PATH = PROJECT_ROOT / 'outputs' / 'figures'
TABLES_PATH = PROJECT_ROOT / 'outputs' / 'tables'
DOCS_PATH = PROJECT_ROOT / 'outputs' / 'generated_docs'
for folder in (FIGURES_PATH, TABLES_PATH, DOCS_PATH):
    folder.mkdir(parents=True, exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')
print(f'Raw dataset: {RAW_PATH}')

Project root: D:\Year3\Retail_Separate_Notebooks
Raw dataset: D:\Year3\Retail_Separate_Notebooks\data\raw\Online Retail.xlsx


## 1. Dataset loading and fingerprint

The fingerprint and structural summary are stored so future runs can confirm that the same source data was analyzed.

In [2]:
if not RAW_PATH.exists():
    raise FileNotFoundError(f'Missing dataset: {RAW_PATH}')

file_hash = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
df = pd.read_excel(RAW_PATH, sheet_name=0)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

dataset_metadata = {
    'source_file': str(RAW_PATH.relative_to(PROJECT_ROOT)),
    'sha256': file_hash,
    'rows': int(df.shape[0]),
    'columns': int(df.shape[1]),
    'column_names': df.columns.tolist(),
    'date_min': str(df['InvoiceDate'].min()),
    'date_max': str(df['InvoiceDate'].max()),
}
print(json.dumps(dataset_metadata, indent=2))
df.info()

{
  "source_file": "data\\raw\\Online Retail.xlsx",
  "sha256": "43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d",
  "rows": 541909,
  "columns": 8,
  "column_names": [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country"
  ],
  "date_min": "2010-12-01 08:26:00",
  "date_max": "2011-12-09 12:50:00"
}
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str          

## 2. Data dictionary

The table distinguishes transaction-level meaning from later customer-level modeling roles.

In [3]:
data_dictionary = pd.DataFrame([
    ['InvoiceNo', 'string', 'Unique transaction identifier', "Some values start with 'C' and indicate cancellations", 'Identifier / transaction grouping'],
    ['StockCode', 'string', 'Product SKU', 'Descriptions may vary for one code', 'Product aggregation'],
    ['Description', 'string', 'Product name', 'Missing values occur for some records', 'Optional product context'],
    ['Quantity', 'integer', 'Items purchased', 'Negative values represent returns; extreme values exist', 'Aggregated behavioral feature'],
    ['InvoiceDate', 'datetime', 'Transaction timestamp', 'Must be validated for range and gaps', 'Temporal features and split design'],
    ['UnitPrice', 'float', 'Price per item', 'Zeros and negative values are invalid for revenue', 'Aggregated monetary feature'],
    ['CustomerID', 'float', 'Customer identifier', 'Approximately 25% missing', 'Customer-level join key'],
    ['Country', 'string', 'Customer country', 'Many countries with uneven frequency', 'Context and optional encoded feature'],
], columns=['Variable', 'Type', 'Business Meaning', 'Quality Issues', 'Modeling Role'])
data_dictionary.to_csv(TABLES_PATH / 'data_dictionary.csv', index=False)
data_dictionary.to_markdown(DOCS_PATH / 'data_dictionary.md', index=False)
data_dictionary

,Variable,Type,Business Meaning,Quality Issues,Modeling Role
0,InvoiceNo,string,Unique transaction identifier,Some values start with 'C' and indicate cancel...,Identifier / transaction grouping
1,StockCode,string,Product SKU,Descriptions may vary for one code,Product aggregation
2,Description,string,Product name,Missing values occur for some records,Optional product context
3,Quantity,integer,Items purchased,Negative values represent returns; extreme val...,Aggregated behavioral feature
4,InvoiceDate,datetime,Transaction timestamp,Must be validated for range and gaps,Temporal features and split design
5,UnitPrice,float,Price per item,Zeros and negative values are invalid for revenue,Aggregated monetary feature
6,CustomerID,float,Customer identifier,Approximately 25% missing,Customer-level join key
7,Country,string,Customer country,Many countries with uneven frequency,Context and optional encoded feature


## 3. Mandatory data quality assessment

Every issue is quantified here before preprocessing decisions are made.

In [4]:
quality_rows = []
def add_quality_issue(name, severity, count, impact, handling):
    quality_rows.append({
        'Issue': name,
        'Severity': severity,
        'Count': int(count),
        'Percentage of rows': round(float(count) / len(df) * 100, 3),
        'Business impact': impact,
        'Proposed handling': handling,
    })

missing_counts = df.isna().sum()
for column, count in missing_counts.items():
    if count:
        severity = 'High' if column == 'CustomerID' else 'Medium'
        impact = 'Prevents reliable customer-level attribution' if column == 'CustomerID' else 'Reduces completeness of analysis'
        handling = 'Drop from customer-level modeling; retain in transaction audit' if column == 'CustomerID' else 'Investigate and document treatment'
        add_quality_issue(f'Missing {column}', severity, count, impact, handling)

add_quality_issue('Exact duplicate rows', 'Medium', df.duplicated().sum(), 'May inflate transaction counts and revenue', 'Remove exact duplicates only')
add_quality_issue('Negative Quantity', 'Medium', (df['Quantity'] < 0).sum(), 'Represents returns or cancellations and affects net value', 'Investigate and net or retain as return behavior')
add_quality_issue('Zero or negative UnitPrice', 'High', (df['UnitPrice'] <= 0).sum(), 'Creates invalid or misleading revenue', 'Remove from monetary calculations after investigation')
add_quality_issue('Cancelled invoice number', 'Medium', df['InvoiceNo'].astype(str).str.startswith('C').sum(), 'Signals cancelled transactions and customer dissatisfaction', 'Track and net where possible')
quantity_low, quantity_high = df['Quantity'].quantile([0.25, 0.75])
quantity_iqr = quantity_high - quantity_low
quantity_outliers = ((df['Quantity'] < quantity_low - 1.5 * quantity_iqr) | (df['Quantity'] > quantity_high + 1.5 * quantity_iqr)).sum()
add_quality_issue('Quantity IQR outliers', 'Medium', quantity_outliers, 'Extreme orders can dominate aggregate features', 'Inspect visually; cap only for modeling if justified')
price_low, price_high = df['UnitPrice'].quantile([0.25, 0.75])
price_iqr = price_high - price_low
price_outliers = ((df['UnitPrice'] < price_low - 1.5 * price_iqr) | (df['UnitPrice'] > price_high + 1.5 * price_iqr)).sum()
add_quality_issue('UnitPrice IQR outliers', 'Medium', price_outliers, 'Extreme prices can distort monetary features', 'Inspect visually; cap only for modeling if justified')

quality_report = pd.DataFrame(quality_rows)
quality_report.to_csv(TABLES_PATH / 'data_quality_report.csv', index=False)
quality_report

,Issue,Severity,Count,Percentage of rows,Business impact,Proposed handling
0,Missing Description,Medium,1454,0.268,Reduces completeness of analysis,Investigate and document treatment
1,Missing CustomerID,High,135080,24.927,Prevents reliable customer-level attribution,Drop from customer-level modeling; retain in t...
2,Exact duplicate rows,Medium,5268,0.972,May inflate transaction counts and revenue,Remove exact duplicates only
3,Negative Quantity,Medium,10624,1.960,Represents returns or cancellations and affect...,Investigate and net or retain as return behavior
4,Zero or negative UnitPrice,High,2517,0.464,Creates invalid or misleading revenue,Remove from monetary calculations after invest...
5,Cancelled invoice number,Medium,9288,1.714,Signals cancelled transactions and customer di...,Track and net where possible
6,Quantity IQR outliers,Medium,58619,10.817,Extreme orders can dominate aggregate features,Inspect visually; cap only for modeling if jus...
7,UnitPrice IQR outliers,Medium,39627,7.312,Extreme prices can distort monetary features,Inspect visually; cap only for modeling if jus...


In [5]:
date_summary = pd.DataFrame({
    'metric': [
        'minimum timestamp',
        'maximum timestamp',
        'unique dates',
        'unique countries',
        'unique invoices',
        'unique products'
    ],
    'value': [
        df['InvoiceDate'].min(),
        df['InvoiceDate'].max(),
        df['InvoiceDate'].dt.date.nunique(),
        df['Country'].nunique(),
        df['InvoiceNo'].nunique(),
        df['StockCode'].nunique()
    ]
})

date_summary

,metric,value
0,minimum timestamp,2010-12-01 08:26:00
1,maximum timestamp,2011-12-09 12:50:00
2,unique dates,305
3,unique countries,38
4,unique invoices,25900
5,unique products,4070


In [6]:
# Build the evidence register used by the insight log
work = df.copy()
work["LineRevenue"] = work["Quantity"] * work["UnitPrice"]

known_customers = work.dropna(subset=["CustomerID"]).copy()
customer_summary = known_customers.groupby("CustomerID").agg(
    invoices=("InvoiceNo", "nunique"),
    monetary=("LineRevenue", "sum"),
    last_purchase=("InvoiceDate", "max"),
    product_breadth=("StockCode", "nunique"),
)

peak_month = (
    work.dropna(subset=["InvoiceDate"])
        .groupby(work["InvoiceDate"].dt.to_period("M"))
        .size()
        .idxmax()
)

repeat_pct = (
    (customer_summary["invoices"] > 1).mean() * 100
    if not customer_summary.empty else 0
)

spend = customer_summary["monetary"].clip(lower=0).sort_values(ascending=False)
top_decile_count = max(1, int(np.ceil(len(spend) * 0.10))) if len(spend) else 0
top_decile_share = (
    spend.head(top_decile_count).sum() / spend.sum() * 100
    if top_decile_count and spend.sum() else 0
)

recency_days = (
    work["InvoiceDate"].max() - customer_summary["last_purchase"]
).dt.days.dropna()

positive_sales = work[work["Quantity"].gt(0) & work["UnitPrice"].gt(0)]
country_sales = positive_sales.groupby("Country")["LineRevenue"].sum()
top_country = country_sales.idxmax() if not country_sales.empty else "Unknown"
top_country_share = (
    country_sales.max() / country_sales.sum() * 100
    if not country_sales.empty and country_sales.sum() else 0
)

sku_description_counts = work.dropna(subset=["Description"]).groupby("StockCode")[
    "Description"
].nunique()

evidence_register = pd.DataFrame([
    {
        "Evidence": f"Peak transaction month: {peak_month}; transaction counts vary by month.",
        "Modeling implication": "Use time-aware features and splits; account for seasonal demand."
    },
    {
        "Evidence": f"{(work['Quantity'] < 0).sum():,} rows have negative quantities.",
        "Modeling implication": "Identify returns and cancellations before calculating customer spend."
    },
    {
        "Evidence": f"{work['CustomerID'].isna().mean() * 100:.1f}% of rows have no CustomerID.",
        "Modeling implication": "Exclude rows without a customer ID from customer-level features."
    },
    {
        "Evidence": f"{repeat_pct:.1f}% of identified customers have more than one invoice.",
        "Modeling implication": "Separate repeat customers from one-time customers when building segments."
    },
    {
        "Evidence": f"The top 10% of identified customers account for {top_decile_share:.1f}% of non-negative customer spend.",
        "Modeling implication": "Consider spend concentration when selecting customer-value measures."
    },
    {
        "Evidence": (
            f"Customer recency has a median of {recency_days.median():.0f} days "
            f"and an interquartile range of {recency_days.quantile(.75) - recency_days.quantile(.25):.0f} days."
            if not recency_days.empty else "Customer recency could not be calculated."
        ),
        "Modeling implication": "Use recency as a customer feature and define the observation cutoff consistently."
    },
    {
        "Evidence": (
            f"Customers have a median product breadth of "
            f"{customer_summary['product_breadth'].median():.0f} distinct stock codes."
            if not customer_summary.empty else "Customer product breadth could not be calculated."
        ),
        "Modeling implication": "Use distinct products purchased as a behavioral feature."
    },
    {
        "Evidence": f"{top_country} contributes {top_country_share:.1f}% of positive line revenue.",
        "Modeling implication": "Check whether geographic concentration affects market comparisons."
    },
    {
        "Evidence": f"{work['InvoiceNo'].astype(str).str.startswith('C').sum():,} rows have cancellation-style invoice identifiers.",
        "Modeling implication": "Treat cancellation records explicitly during revenue and behavior calculations."
    },
    {
        "Evidence": f"{(sku_description_counts > 1).sum():,} stock codes have multiple recorded descriptions.",
        "Modeling implication": "Use StockCode as the product key; treat descriptions as labels."
    },
])

evidence_register.to_csv(TABLES_PATH / "eda_evidence_register.csv", index=False)
evidence_register

,Evidence,Modeling implication
0,Peak transaction month: 2011-11; transaction c...,Use time-aware features and splits; account fo...
1,"10,624 rows have negative quantities.",Identify returns and cancellations before calc...
2,24.9% of rows have no CustomerID.,Exclude rows without a customer ID from custom...
3,70.0% of identified customers have more than o...,Separate repeat customers from one-time custom...
4,The top 10% of identified customers account fo...,Consider spend concentration when selecting cu...
5,Customer recency has a median of 49 days and a...,Use recency as a customer feature and define t...
6,Customers have a median product breadth of 35 ...,Use distinct products purchased as a behaviora...
7,United Kingdom contributes 84.6% of positive l...,Check whether geographic concentration affects...
8,"9,288 rows have cancellation-style invoice ide...",Treat cancellation records explicitly during r...
9,650 stock codes have multiple recorded descrip...,Use StockCode as the product key; treat descri...


## Phase 1 outputs

Saved outputs include `data_dictionary.md`, `data_dictionary.csv`, `data_quality_report.csv`, `eda_evidence_register.csv`, and the required EDA figures under `outputs/figures/`. The next notebook will use these findings to justify cleaning, temporal windows, target construction, and customer-level features.